## CSC 4792 MINI PROJECT: Mumbwa Town Council CDF Dataset

This notebook documents the full CDF data pipeline for Mumbwa Town Council.


##  Installing Required Packages



In [1]:
%pip install pandas requests beautifulsoup4 pdfplumber


Note: you may need to restart the kernel to use updated packages.


## Importing Libraries And Setting Project Folders

The project stores the CDF workflow under `data/CDF`. These paths are defined once and reused throughout the notebook.


In [2]:
from pathlib import Path
import sys

import pandas as pd
import pdfplumber
import requests
import urllib3
from bs4 import BeautifulSoup
from urllib.parse import urljoin


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)

    for candidate in [start, *start.parents]:
        if (candidate / "src").is_dir() and (candidate / "data").is_dir():
            return candidate

    fallback = Path(r"C:\Users\Moses Chaswala\Desktop\Mumbwa-Dataset\mumbwa-town-council-dataset")
    if fallback.exists():
        return fallback

    raise FileNotFoundError("Could not locate the project root folder.")


project_root = find_project_root()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.collector import CDF_TRACKER_URL, EXPECTED_CDF_LINKS, find_pdf_links, save_source_list, download_pdfs
from src.extractor import extract_all_reconstructed_pdfs
from src.cleaner import clean_csv_file
from src.integrator import create_final_datasets

cdf_root = project_root / "data" / "CDF"
raw_dir = cdf_root / "raw"
raw_pdf_dir = raw_dir / "pdfs"
source_list_file = raw_dir / "cdf_pdf_sources.csv"
intermediate_dir = cdf_root / "intermediate"
reconstructed_pdf_dir = intermediate_dir / "reconstructed_pdfs"
extracted_dir = cdf_root / "extracted"
processed_dir = cdf_root / "processed"
final_dir = cdf_root / "final"

for folder in [raw_pdf_dir, reconstructed_pdf_dir, extracted_dir, processed_dir, final_dir]:
    folder.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_rows", 50)
pd.set_option("display.max_columns", None)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

pd.set_option("display.max_colwidth", None)

print("Project root:", project_root)
print("CDF data root:", cdf_root)


Project root: C:\Users\Moses Chaswala\Desktop\Mumbwa-Dataset\mumbwa-town-council-dataset
CDF data root: C:\Users\Moses Chaswala\Desktop\Mumbwa-Dataset\mumbwa-town-council-dataset\data\CDF


##  Mumbwa Town Council CDF Tracker Page

The CDF Tracker page is the source page used to discover the official PDF documents.


In [3]:
council_name = "Mumbwa Town Council"
base_url = "https://www.mumbwacouncil.gov.zm/"

print(council_name)
print("Base website:", base_url)
print("CDF tracker:", CDF_TRACKER_URL)


Mumbwa Town Council
Base website: https://www.mumbwacouncil.gov.zm/
CDF tracker: https://www.mumbwacouncil.gov.zm/?page_id=932


##  Web Scraping: Finding CDF PDF Links

This cell scrapes the CDF Tracker page for official CDF PDF links. If the site is unavailable, it falls back to the saved source inventory so the rest of the notebook can still run from local files.


In [4]:
def find_pdf_links_without_certificate_check(page_url=CDF_TRACKER_URL):
    response = requests.get(page_url, timeout=30, verify=False)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")
    pdf_links = []

    for link in soup.find_all("a", href=True):
        href = link["href"]
        text = link.get_text(" ", strip=True)

        if ".pdf" not in href.lower():
            continue

        full_url = urljoin(page_url, href)
        file_name = full_url.split("/")[-1]

        if file_name in EXPECTED_CDF_LINKS:
            details = EXPECTED_CDF_LINKS[file_name]
            pdf_links.append({
                "source_id": details["source_id"],
                "document": file_name,
                "category": details["category"],
                "constituency": details["constituency"],
                "year": details["year"],
                "extraction_method": "recreated_pdf",
                "title_on_site": text,
                "source_url": full_url,
                "tracker_page": page_url,
            })

    return pdf_links


try:
    pdf_links = find_pdf_links(CDF_TRACKER_URL)
    source_df = pd.DataFrame(pdf_links)
    print("CDF PDF links scraped from website:", len(source_df))
except Exception as exc:
    print("Default scrape could not verify the CDF Tracker page certificate.")
    print("Reason:", exc)

    try:
        pdf_links = find_pdf_links_without_certificate_check(CDF_TRACKER_URL)
        source_df = pd.DataFrame(pdf_links)
        print("CDF PDF links scraped with certificate verification disabled:", len(source_df))
    except Exception as fallback_exc:
        print("Could not scrape the CDF Tracker page with fallback settings.")
        print("Reason:", fallback_exc)

        if source_list_file.exists():
            source_df = pd.read_csv(source_list_file, sep="|")
            pdf_links = source_df.to_dict("records")
            print("Loaded saved source inventory:", len(source_df))
        else:
            source_df = pd.DataFrame()
            pdf_links = []
            print("No saved source inventory was found.")

source_df


Default scrape could not verify the CDF Tracker page certificate.
Reason: HTTPSConnectionPool(host='www.mumbwacouncil.gov.zm', port=443): Max retries exceeded with url: /?page_id=932 (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1081)')))
CDF PDF links scraped with certificate verification disabled: 10


,source_id,document,category,constituency,year,extraction_method,title_on_site,source_url,tracker_page
0,cdf_community_projects_mumbwa_2025,2025-Approved-Community-Projects-Mumbwa-Central.pdf,Community Projects,Mumbwa,2025,recreated_pdf,2025 Approved Community Projects-Mumbwa Central,https://www.mumbwacouncil.gov.zm/wp-content/uploads/2025/06/2025-Approved-Community-Projects-Mumbwa-Central.pdf,https://www.mumbwacouncil.gov.zm/?page_id=932
1,cdf_not_approved_projects_mumbwa_2025,NOT-APPROVED-COMMUNITY-PROJECTS-MUMBWA.pdf,Not Approved Community Projects,Mumbwa,2025,recreated_pdf,2025 Proposed Community Projects-Mumbwa Central,https://www.mumbwacouncil.gov.zm/wp-content/uploads/2025/12/NOT-APPROVED-COMMUNITY-PROJECTS-MUMBWA.pdf,https://www.mumbwacouncil.gov.zm/?page_id=932
2,cdf_skills_bursaries_mumbwa_2025,2025-Approved-CDF-Skills-Development-Bursaries-for-Mumbwa-Constituency.pdf,Skills Bursaries,Mumbwa,2025,recreated_pdf,2025 Approved CDF Skills Development Bursaries for Mumbwa Constituency,https://www.mumbwacouncil.gov.zm/wp-content/uploads/2025/09/2025-Approved-CDF-Skills-Development-Bursaries-for-Mumbwa-Constituency.pdf,https://www.mumbwacouncil.gov.zm/?page_id=932
3,cdf_secondary_bursaries_mumbwa_2025,2025-Approved-CDF-Secondary-Boarding-School-Bursaries-for-Mumbwa-Constituency.pdf,Secondary Bursaries,Mumbwa,2025,recreated_pdf,2025 Approved CDF Secondary Boarding School Bursaries for Mumbwa Constituency,https://www.mumbwacouncil.gov.zm/wp-content/uploads/2025/09/2025-Approved-CDF-Secondary-Boarding-School-Bursaries-for-Mumbwa-Constituency.pdf,https://www.mumbwacouncil.gov.zm/?page_id=932
4,cdf_grants_mumbwa_2025,2025-Approved-CDF-Grants-for-Mumbwa-Constituency.pdf,Grants,Mumbwa,2025,recreated_pdf,2025 Approved CDF Empowerment Grants for Mumbwa Constituency,https://www.mumbwacouncil.gov.zm/wp-content/uploads/2025/09/2025-Approved-CDF-Grants-for-Mumbwa-Constituency.pdf,https://www.mumbwacouncil.gov.zm/?page_id=932
5,cdf_community_projects_nangoma_2025,2025-Approved-Community-Projects-Nangoma-Constituency.pdf,Community Projects,Nangoma,2025,recreated_pdf,2025 Approved Community Projects-Nangoma Constituency,https://www.mumbwacouncil.gov.zm/wp-content/uploads/2025/06/2025-Approved-Community-Projects-Nangoma-Constituency.pdf,https://www.mumbwacouncil.gov.zm/?page_id=932
6,cdf_not_approved_projects_nangoma_2025,NOT-APPROVED-COMMUNITY-PROJECTS-NANGOMA.pdf,Not Approved Community Projects,Nangoma,2025,recreated_pdf,2025 Proposed Community Projects-Nangoma Constituency,https://www.mumbwacouncil.gov.zm/wp-content/uploads/2025/12/NOT-APPROVED-COMMUNITY-PROJECTS-NANGOMA.pdf,https://www.mumbwacouncil.gov.zm/?page_id=932
7,cdf_grants_nangoma_2025,2025-Approve-CDF-Grants-for-Nangoma-Constituency.pdf,Grants,Nangoma,2025,recreated_pdf,2025 Approved CDF Empowerment Grants for Nangoma Constituency,https://www.mumbwacouncil.gov.zm/wp-content/uploads/2025/09/2025-Approve-CDF-Grants-for-Nangoma-Constituency.pdf,https://www.mumbwacouncil.gov.zm/?page_id=932
8,cdf_skills_bursaries_nangoma_2025,2025-Approved-CDF-Skills-Development-Bursaries-for-Nangoma-Constituency.pdf,Skills Bursaries,Nangoma,2025,recreated_pdf,2025 Approved CDF Skills Development Bursaries for Nangoma Constituency,https://www.mumbwacouncil.gov.zm/wp-content/uploads/2025/09/2025-Approved-CDF-Skills-Development-Bursaries-for-Nangoma-Constituency.pdf,https://www.mumbwacouncil.gov.zm/?page_id=932
9,cdf_secondary_bursaries_nangoma_2025,2025-Approved-CDF-Secondary-Boarding-School-Bursaries-for-Nangoma-Constituency.pdf,Secondary Bursaries,Nangoma,2025,recreated_pdf,2025 Approved CDF Secondary Boarding School Bursaries for Nangoma Constituency,https://www.mumbwacouncil.gov.zm/wp-content/uploads/2025/09/2025-Approved-CDF-Secondary-Boarding-School-Bursaries-for-Nangoma-Constituency.pdf,https://www.mumbwacouncil.gov.zm/?page_id=932


## Saving Source Inventory CSV

The source inventory records the PDF filename, source URL, category, constituency, and year before any extraction happens.


In [5]:
source_inventory_columns = [
    "source_id",
    "document",
    "category",
    "constituency",
    "year",
    "extraction_method",
    "source_url",
    "tracker_page",
]

if pdf_links:
    source_df_to_save = source_df.reindex(columns=source_inventory_columns)
    saved_source_file = save_source_list(
        source_df_to_save.to_dict("records"),
        source_list_file,
    )
    print("Saved source inventory to:", saved_source_file)
else:
    print("No PDF links available to save.")

if source_list_file.exists():
    source_inventory = pd.read_csv(source_list_file, sep="|")
    display(source_inventory)
else:
    source_inventory = pd.DataFrame()
    print("Source inventory file does not exist yet.")


Saved source inventory to: C:\Users\Moses Chaswala\Desktop\Mumbwa-Dataset\mumbwa-town-council-dataset\data\CDF\raw\cdf_pdf_sources.csv


,source_id,document,category,constituency,year,extraction_method,source_url,tracker_page
0,cdf_community_projects_mumbwa_2025,2025-Approved-Community-Projects-Mumbwa-Central.pdf,Community Projects,Mumbwa,2025,recreated_pdf,https://www.mumbwacouncil.gov.zm/wp-content/uploads/2025/06/2025-Approved-Community-Projects-Mumbwa-Central.pdf,https://www.mumbwacouncil.gov.zm/?page_id=932
1,cdf_not_approved_projects_mumbwa_2025,NOT-APPROVED-COMMUNITY-PROJECTS-MUMBWA.pdf,Not Approved Community Projects,Mumbwa,2025,recreated_pdf,https://www.mumbwacouncil.gov.zm/wp-content/uploads/2025/12/NOT-APPROVED-COMMUNITY-PROJECTS-MUMBWA.pdf,https://www.mumbwacouncil.gov.zm/?page_id=932
2,cdf_skills_bursaries_mumbwa_2025,2025-Approved-CDF-Skills-Development-Bursaries-for-Mumbwa-Constituency.pdf,Skills Bursaries,Mumbwa,2025,recreated_pdf,https://www.mumbwacouncil.gov.zm/wp-content/uploads/2025/09/2025-Approved-CDF-Skills-Development-Bursaries-for-Mumbwa-Constituency.pdf,https://www.mumbwacouncil.gov.zm/?page_id=932
3,cdf_secondary_bursaries_mumbwa_2025,2025-Approved-CDF-Secondary-Boarding-School-Bursaries-for-Mumbwa-Constituency.pdf,Secondary Bursaries,Mumbwa,2025,recreated_pdf,https://www.mumbwacouncil.gov.zm/wp-content/uploads/2025/09/2025-Approved-CDF-Secondary-Boarding-School-Bursaries-for-Mumbwa-Constituency.pdf,https://www.mumbwacouncil.gov.zm/?page_id=932
4,cdf_grants_mumbwa_2025,2025-Approved-CDF-Grants-for-Mumbwa-Constituency.pdf,Grants,Mumbwa,2025,recreated_pdf,https://www.mumbwacouncil.gov.zm/wp-content/uploads/2025/09/2025-Approved-CDF-Grants-for-Mumbwa-Constituency.pdf,https://www.mumbwacouncil.gov.zm/?page_id=932
5,cdf_community_projects_nangoma_2025,2025-Approved-Community-Projects-Nangoma-Constituency.pdf,Community Projects,Nangoma,2025,recreated_pdf,https://www.mumbwacouncil.gov.zm/wp-content/uploads/2025/06/2025-Approved-Community-Projects-Nangoma-Constituency.pdf,https://www.mumbwacouncil.gov.zm/?page_id=932
6,cdf_not_approved_projects_nangoma_2025,NOT-APPROVED-COMMUNITY-PROJECTS-NANGOMA.pdf,Not Approved Community Projects,Nangoma,2025,recreated_pdf,https://www.mumbwacouncil.gov.zm/wp-content/uploads/2025/12/NOT-APPROVED-COMMUNITY-PROJECTS-NANGOMA.pdf,https://www.mumbwacouncil.gov.zm/?page_id=932
7,cdf_grants_nangoma_2025,2025-Approve-CDF-Grants-for-Nangoma-Constituency.pdf,Grants,Nangoma,2025,recreated_pdf,https://www.mumbwacouncil.gov.zm/wp-content/uploads/2025/09/2025-Approve-CDF-Grants-for-Nangoma-Constituency.pdf,https://www.mumbwacouncil.gov.zm/?page_id=932
8,cdf_skills_bursaries_nangoma_2025,2025-Approved-CDF-Skills-Development-Bursaries-for-Nangoma-Constituency.pdf,Skills Bursaries,Nangoma,2025,recreated_pdf,https://www.mumbwacouncil.gov.zm/wp-content/uploads/2025/09/2025-Approved-CDF-Skills-Development-Bursaries-for-Nangoma-Constituency.pdf,https://www.mumbwacouncil.gov.zm/?page_id=932
9,cdf_secondary_bursaries_nangoma_2025,2025-Approved-CDF-Secondary-Boarding-School-Bursaries-for-Nangoma-Constituency.pdf,Secondary Bursaries,Nangoma,2025,recreated_pdf,https://www.mumbwacouncil.gov.zm/wp-content/uploads/2025/09/2025-Approved-CDF-Secondary-Boarding-School-Bursaries-for-Nangoma-Constituency.pdf,https://www.mumbwacouncil.gov.zm/?page_id=932


## Downloading Raw PDFs

The downloader saves official PDFs into `data/CDF/raw/pdfs`. Existing PDFs are not overwritten


In [6]:
if pdf_links:
    try:
        downloaded_files = download_pdfs(pdf_links, raw_pdf_dir)
    except Exception as exc:
        print("PDF download step could not complete.")
        print("Reason:", exc)
        downloaded_files = sorted(raw_pdf_dir.glob("*.pdf"))
else:
    downloaded_files = sorted(raw_pdf_dir.glob("*.pdf"))

raw_pdf_files = sorted(raw_pdf_dir.glob("*.pdf"))
print("Raw PDFs available:", len(raw_pdf_files))
for pdf_file in raw_pdf_files:
    print(pdf_file.name)


Already exists, skipping: C:\Users\Moses Chaswala\Desktop\Mumbwa-Dataset\mumbwa-town-council-dataset\data\CDF\raw\pdfs\2025-Approved-Community-Projects-Mumbwa-Central.pdf
Already exists, skipping: C:\Users\Moses Chaswala\Desktop\Mumbwa-Dataset\mumbwa-town-council-dataset\data\CDF\raw\pdfs\NOT-APPROVED-COMMUNITY-PROJECTS-MUMBWA.pdf
Already exists, skipping: C:\Users\Moses Chaswala\Desktop\Mumbwa-Dataset\mumbwa-town-council-dataset\data\CDF\raw\pdfs\2025-Approved-CDF-Skills-Development-Bursaries-for-Mumbwa-Constituency.pdf
Already exists, skipping: C:\Users\Moses Chaswala\Desktop\Mumbwa-Dataset\mumbwa-town-council-dataset\data\CDF\raw\pdfs\2025-Approved-CDF-Secondary-Boarding-School-Bursaries-for-Mumbwa-Constituency.pdf
Already exists, skipping: C:\Users\Moses Chaswala\Desktop\Mumbwa-Dataset\mumbwa-town-council-dataset\data\CDF\raw\pdfs\2025-Approved-CDF-Grants-for-Mumbwa-Constituency.pdf
Already exists, skipping: C:\Users\Moses Chaswala\Desktop\Mumbwa-Dataset\mumbwa-town-council-dataset

## Checking For PDF Table Availability

This quick check counts extractable tables in each raw PDF.


In [7]:
table_summary = []

for pdf_file in raw_pdf_files:
    try:
        with pdfplumber.open(pdf_file) as pdf:
            page_count = len(pdf.pages)
            table_count = sum(len(page.extract_tables() or []) for page in pdf.pages)
    except Exception as exc:
        page_count = pd.NA
        table_count = pd.NA
        print(f"Could not inspect {pdf_file.name}: {exc}")

    table_summary.append({
        "document": pdf_file.name,
        "pages": page_count,
        "tables_detected": table_count,
    })

raw_pdf_summary = pd.DataFrame(table_summary)
raw_pdf_summary


,document,pages,tables_detected
0,2025-Approve-CDF-Grants-for-Nangoma-Constituency.pdf,3,0
1,2025-Approved-CDF-Grants-for-Mumbwa-Constituency.pdf,3,0
2,2025-Approved-CDF-Secondary-Boarding-School-Bursaries-for-Mumbwa-Constituency.pdf,16,0
3,2025-Approved-CDF-Secondary-Boarding-School-Bursaries-for-Nangoma-Constituency.pdf,13,0
4,2025-Approved-CDF-Skills-Development-Bursaries-for-Mumbwa-Constituency.pdf,11,0
5,2025-Approved-CDF-Skills-Development-Bursaries-for-Nangoma-Constituency.pdf,13,0
6,2025-Approved-Community-Projects-Mumbwa-Central.pdf,2,0
7,2025-Approved-Community-Projects-Nangoma-Constituency.pdf,3,0
8,NOT-APPROVED-COMMUNITY-PROJECTS-MUMBWA.pdf,4,0
9,NOT-APPROVED-COMMUNITY-PROJECTS-NANGOMA.pdf,7,0


## Extract Tables From Reconstructed PDFs

The source PDFs contain tables that are difficult to extract consistently. The reconstructed PDFs in `data/CDF/intermediate/reconstructed_pdfs` are used as the extraction-ready inputs, and the extracted CSV files are saved in `data/CDF/extracted`.


In [8]:
reconstructed_files = sorted(reconstructed_pdf_dir.glob("*.pdf"))
print("Reconstructed PDFs available:", len(reconstructed_files))
for pdf_file in reconstructed_files:
    print(pdf_file.name)


Reconstructed PDFs available: 10
2025-Approve-CDF-Grants-for-Nangoma-Constituency-Recreated_tables.pdf
2025-Approved-CDF-Grants-for-Mumbwa-Constituency-Recreated_tables.pdf
2025-Approved-CDF-Secondary-Boarding-School-Bursaries-for-Mumbwa-Constituency-Recreated_tables.pdf
2025-Approved-CDF-Secondary-Boarding-School-Bursaries-for-Nangoma-Constituency-Recreated_tables.pdf
2025-Approved-CDF-Skills-Development-Bursaries-for-Mumbwa-Constituency-Recreated_tables.pdf
2025-Approved-CDF-Skills-Development-Bursaries-for-Nangoma-Constituency-Recreated_tables.pdf
2025_Approved_Community_Projects_Mumbwa_Central_Recreated_Tables.pdf
2025_Approved_Community_Projects_Nangoma_Constituency_Recreated_Tables.pdf
NOT-APPROVED-COMMUNITY-PROJECTS-MUMBWA-Recreated-Tables.pdf
NOT-APPROVED-COMMUNITY-PROJECTS-NANGOMA-Recreated-Tables.pdf


In [9]:
existing_extracted_files = sorted(extracted_dir.glob("*.csv"))

if existing_extracted_files:
    print("Using existing extracted CSV files:", len(existing_extracted_files))
    new_extracted_files = []
else:
    new_extracted_files = extract_all_reconstructed_pdfs(
        input_dir=reconstructed_pdf_dir,
        output_dir=extracted_dir,
    )

extracted_files = sorted(extracted_dir.glob("*.csv"))
print("Extracted CSV files available:", len(extracted_files))
for csv_file in extracted_files:
    print(csv_file.name)


Using existing extracted CSV files: 10
Extracted CSV files available: 10
2025_mumbwa_community_projects.csv
2025_mumbwa_grants.csv
2025_mumbwa_not_approved_community_projects.csv
2025_mumbwa_secondary_bursaries.csv
2025_mumbwa_skills_bursaries.csv
2025_nangoma_community_projects.csv
2025_nangoma_grants.csv
2025_nangoma_not_approved_community_projects.csv
2025_nangoma_secondary_bursaries.csv
2025_nangoma_skills_bursaries.csv


##  Preview Extracted CSV Files

These are the raw table extracts before cleaning and standardizing column names.


In [10]:
extracted_summary = []

for csv_file in extracted_files:
    df = pd.read_csv(csv_file, sep="|")
    extracted_summary.append({
        "file": csv_file.name,
        "rows": len(df),
        "columns": len(df.columns),
    })

extracted_summary_df = pd.DataFrame(extracted_summary)
extracted_summary_df


,file,rows,columns
0,2025_mumbwa_community_projects.csv,14,17
1,2025_mumbwa_grants.csv,67,18
2,2025_mumbwa_not_approved_community_projects.csv,59,16
3,2025_mumbwa_secondary_bursaries.csv,493,24
4,2025_mumbwa_skills_bursaries.csv,399,20
5,2025_nangoma_community_projects.csv,16,18
6,2025_nangoma_grants.csv,72,17
7,2025_nangoma_not_approved_community_projects.csv,76,19
8,2025_nangoma_secondary_bursaries.csv,294,19
9,2025_nangoma_skills_bursaries.csv,381,24


In [11]:
if extracted_files:
    sample_extracted_file = extracted_files[0]
    sample_extracted_df = pd.read_csv(sample_extracted_file, sep="|")
    print("Previewing:", sample_extracted_file.name)
    display(sample_extracted_df.head())
else:
    print("No extracted CSV files found.")


Previewing: 2025_mumbwa_community_projects.csv


,no,project_name,sector,type_of_project,district,constituency,ward,project_site_location,year_funded,work_package_inclusive_of_cdf_branding,comments_remarks,page_number,table_number,source_document,year,cdf_category,extraction_method
0,1,Construction of 1x2 Classroom Block at Malombe Primary School,Education,Construction,Mumbwa,Mumbwa,Shimbizhi,Malombe Primary School,2025,1x2 Classroom Block,Approved,1,1,2025_Approved_Community_Projects_Mumbwa_Central_Recreated_Tables.pdf,2025,community_projects,reconstructed_then_pdfplumber
1,2,Construction of 1x2 Classroom Block at Kamilambo Primary School,Education,Construction,Mumbwa,Mumbwa,Kamilambo,Kamilambo,2025,1x2 Classroom Block,Approved,1,1,2025_Approved_Community_Projects_Mumbwa_Central_Recreated_Tables.pdf,2025,community_projects,reconstructed_then_pdfplumber
2,3,Completion of Kabawa Rural Health Post,Health,Completion,Mumbwa,Mumbwa,Chibolyo,Kabawa Rural Health Post,2025,1No. Health Post Incinerator,Approved,1,1,2025_Approved_Community_Projects_Mumbwa_Central_Recreated_Tables.pdf,2025,community_projects,reconstructed_then_pdfplumber
3,4,Completion of a Maternity Wing & Water Reticulation System,Health,Construction & Reticulation,Mumbwa,Mumbwa,Makebo,Kabwanga Rural Health Post,2025,"Completion of Maternity Wing Water reticulation,incinerator, Equipping of facility (8 Beds,4 Chairs,4 Cabinets, 3 operation beds)",Approved,1,1,2025_Approved_Community_Projects_Mumbwa_Central_Recreated_Tables.pdf,2025,community_projects,reconstructed_then_pdfplumber
4,5,Construction of 1x2 Semi detached Teachers Staff House & installation of water reticulation at Kalenda Primary School,Education,Construction,Mumbwa,Mumbwa,Kalwanyembe,Kalenda Secondary School,2025 2025,1No. Semi detached Staff house Water Reticulation,Approved,1,1,2025_Approved_Community_Projects_Mumbwa_Central_Recreated_Tables.pdf,2025,community_projects,reconstructed_then_pdfplumber


## Cleaning Extracted CSV Files

The cleaning step standardizes headers, removes empty and duplicate rows, normalizes category-specific columns, and saves clean CSV files into `data/CDF/processed`.


In [12]:
processed_files = []

for csv_file in extracted_files:
    processed_files.append(clean_csv_file(csv_file, output_dir=processed_dir))

processed_files = sorted(processed_dir.glob("*.csv"))
print("Processed CSV files available:", len(processed_files))
for csv_file in processed_files:
    print(csv_file.name)


Cleaning: 2025_mumbwa_community_projects.csv
Already exists, skipping: C:\Users\Moses Chaswala\Desktop\Mumbwa-Dataset\mumbwa-town-council-dataset\data\CDF\processed\2025_mumbwa_community_projects_clean.csv

Cleaning: 2025_mumbwa_grants.csv
Already exists, skipping: C:\Users\Moses Chaswala\Desktop\Mumbwa-Dataset\mumbwa-town-council-dataset\data\CDF\processed\2025_mumbwa_grants_clean.csv

Cleaning: 2025_mumbwa_not_approved_community_projects.csv
Already exists, skipping: C:\Users\Moses Chaswala\Desktop\Mumbwa-Dataset\mumbwa-town-council-dataset\data\CDF\processed\2025_mumbwa_not_approved_community_projects_clean.csv

Cleaning: 2025_mumbwa_secondary_bursaries.csv
Already exists, skipping: C:\Users\Moses Chaswala\Desktop\Mumbwa-Dataset\mumbwa-town-council-dataset\data\CDF\processed\2025_mumbwa_secondary_bursaries_clean.csv

Cleaning: 2025_mumbwa_skills_bursaries.csv
Already exists, skipping: C:\Users\Moses Chaswala\Desktop\Mumbwa-Dataset\mumbwa-town-council-dataset\data\CDF\processed\2025_

##  Preview Cleaned CSV Files

The processed files are the category-level cleaned datasets used to build the final deliverables.


In [13]:
processed_summary = []

for csv_file in processed_files:
    df = pd.read_csv(csv_file, sep="|")
    processed_summary.append({
        "file": csv_file.name,
        "rows": len(df),
        "columns": len(df.columns),
    })

processed_summary_df = pd.DataFrame(processed_summary)
processed_summary_df


,file,rows,columns
0,2025_mumbwa_community_projects_clean.csv,14,19
1,2025_mumbwa_grants_clean.csv,67,13
2,2025_mumbwa_not_approved_community_projects_clean.csv,59,19
3,2025_mumbwa_secondary_bursaries_clean.csv,493,18
4,2025_mumbwa_skills_bursaries_clean.csv,399,16
5,2025_nangoma_community_projects_clean.csv,16,19
6,2025_nangoma_grants_clean.csv,72,13
7,2025_nangoma_not_approved_community_projects_clean.csv,76,19
8,2025_nangoma_secondary_bursaries_clean.csv,294,18
9,2025_nangoma_skills_bursaries_clean.csv,381,16


In [14]:
if processed_files:
    sample_processed_file = processed_files[0]
    sample_processed_df = pd.read_csv(sample_processed_file, sep="|")
    print("Previewing:", sample_processed_file.name)
    display(sample_processed_df.head())
else:
    print("No processed CSV files found.")


Previewing: 2025_mumbwa_community_projects_clean.csv


,record_number,project_name,project_description,sector,type_of_project,district,constituency,ward,zone,project_site_location,distance_km,year_funded,work_package,scope_of_works,status,comments,source_id,year,cdf_category
0,1,Construction of 1x2 Classroom Block at Malombe Primary School,NaN,Education,Construction,Mumbwa,Mumbwa,Shimbizhi,NaN,Malombe Primary School,NaN,2025,1x2 Classroom Block,NaN,Approved,NaN,cdf_community_projects_mumbwa_2025,2025,community_projects
1,2,Construction of 1x2 Classroom Block at Kamilambo Primary School,NaN,Education,Construction,Mumbwa,Mumbwa,Kamilambo,NaN,Kamilambo,NaN,2025,1x2 Classroom Block,NaN,Approved,NaN,cdf_community_projects_mumbwa_2025,2025,community_projects
2,3,Completion of Kabawa Rural Health Post,NaN,Health,Completion,Mumbwa,Mumbwa,Chibolyo,NaN,Kabawa Rural Health Post,NaN,2025,1No. Health Post Incinerator,NaN,Approved,NaN,cdf_community_projects_mumbwa_2025,2025,community_projects
3,4,Completion of a Maternity Wing & Water Reticulation System,NaN,Health,Construction & Reticulation,Mumbwa,Mumbwa,Makebo,NaN,Kabwanga Rural Health Post,NaN,2025,"Completion of Maternity Wing Water reticulation,incinerator, Equipping of facility (8 Beds,4 Chairs,4 Cabinets, 3 operation beds)",NaN,Approved,NaN,cdf_community_projects_mumbwa_2025,2025,community_projects
4,5,Construction of 1x2 Semi detached Teachers Staff House & installation of water reticulation at Kalenda Primary School,NaN,Education,Construction,Mumbwa,Mumbwa,Kalwanyembe,NaN,Kalenda Secondary School,NaN,2025 2025,1No. Semi detached Staff house Water Reticulation,NaN,Approved,NaN,cdf_community_projects_mumbwa_2025,2025,community_projects


## Create Final CDF Datasets

The final step combines matching processed files into five final datasets: community projects, not-approved community projects, grants, secondary bursaries, and skills bursaries.


In [15]:
final_output_files = create_final_datasets(
    input_dir=processed_dir,
    output_dir=final_dir,
)

final_files = sorted(final_dir.glob("*.csv"))
print("Final CSV files available:", len(final_files))
for csv_file in final_files:
    print(csv_file.name)


Created: C:\Users\Moses Chaswala\Desktop\Mumbwa-Dataset\mumbwa-town-council-dataset\data\CDF\final\db-unza26-csc4792-mumbwa_town_council_cdf_grants.csv
Rows: 139

Created: C:\Users\Moses Chaswala\Desktop\Mumbwa-Dataset\mumbwa-town-council-dataset\data\CDF\final\db-unza26-csc4792-mumbwa_town_council_cdf_community_projects.csv
Rows: 30

Created: C:\Users\Moses Chaswala\Desktop\Mumbwa-Dataset\mumbwa-town-council-dataset\data\CDF\final\db-unza26-csc4792-mumbwa_town_council_cdf_not_approved_community_projects.csv
Rows: 135

Created: C:\Users\Moses Chaswala\Desktop\Mumbwa-Dataset\mumbwa-town-council-dataset\data\CDF\final\db-unza26-csc4792-mumbwa_town_council_cdf_skills_bursaries.csv
Rows: 780

Created: C:\Users\Moses Chaswala\Desktop\Mumbwa-Dataset\mumbwa-town-council-dataset\data\CDF\final\db-unza26-csc4792-mumbwa_town_council_cdf_secondary_bursaries.csv
Rows: 787

Final CSV files available: 5
db-unza26-csc4792-mumbwa_town_council_cdf_community_projects.csv
db-unza26-csc4792-mumbwa_town_co

## Final Data Preview

These are the final project deliverables saved under `data/CDF/final`.


In [16]:
final_summary = []

for csv_file in final_files:
    df = pd.read_csv(csv_file, sep="|")
    final_summary.append({
        "file": csv_file.name,
        "rows": len(df),
        "columns": len(df.columns),
    })

final_summary_df = pd.DataFrame(final_summary)
final_summary_df


,file,rows,columns
0,db-unza26-csc4792-mumbwa_town_council_cdf_community_projects.csv,30,19
1,db-unza26-csc4792-mumbwa_town_council_cdf_grants.csv,139,13
2,db-unza26-csc4792-mumbwa_town_council_cdf_not_approved_community_projects.csv,135,19
3,db-unza26-csc4792-mumbwa_town_council_cdf_secondary_bursaries.csv,787,18
4,db-unza26-csc4792-mumbwa_town_council_cdf_skills_bursaries.csv,780,16


In [17]:
for csv_file in final_files:
    df = pd.read_csv(csv_file, sep="|")
    print()
    print("=" * 100)
    print(csv_file.name)
    print("Rows:", len(df), "Columns:", len(df.columns))
    display(df.head())



db-unza26-csc4792-mumbwa_town_council_cdf_community_projects.csv
Rows: 30 Columns: 19


,record_number,project_name,project_description,sector,type_of_project,district,constituency,ward,zone,project_site_location,distance_km,year_funded,work_package,scope_of_works,status,comments,source_id,year,cdf_category
0,1.0,Construction of 1x2 Classroom Block at Malombe Primary School,NaN,Education,Construction,Mumbwa,Mumbwa,Shimbizhi,NaN,Malombe Primary School,NaN,2025,1x2 Classroom Block,NaN,Approved,NaN,cdf_community_projects_mumbwa_2025,2025,community_projects
1,2.0,Construction of 1x2 Classroom Block at Kamilambo Primary School,NaN,Education,Construction,Mumbwa,Mumbwa,Kamilambo,NaN,Kamilambo,NaN,2025,1x2 Classroom Block,NaN,Approved,NaN,cdf_community_projects_mumbwa_2025,2025,community_projects
2,3.0,Completion of Kabawa Rural Health Post,NaN,Health,Completion,Mumbwa,Mumbwa,Chibolyo,NaN,Kabawa Rural Health Post,NaN,2025,1No. Health Post Incinerator,NaN,Approved,NaN,cdf_community_projects_mumbwa_2025,2025,community_projects
3,4.0,Completion of a Maternity Wing & Water Reticulation System,NaN,Health,Construction & Reticulation,Mumbwa,Mumbwa,Makebo,NaN,Kabwanga Rural Health Post,NaN,2025,"Completion of Maternity Wing Water reticulation,incinerator, Equipping of facility (8 Beds,4 Chairs,4 Cabinets, 3 operation beds)",NaN,Approved,NaN,cdf_community_projects_mumbwa_2025,2025,community_projects
4,5.0,Construction of 1x2 Semi detached Teachers Staff House & installation of water reticulation at Kalenda Primary School,NaN,Education,Construction,Mumbwa,Mumbwa,Kalwanyembe,NaN,Kalenda Secondary School,NaN,2025 2025,1No. Semi detached Staff house Water Reticulation,NaN,Approved,NaN,cdf_community_projects_mumbwa_2025,2025,community_projects



db-unza26-csc4792-mumbwa_town_council_cdf_grants.csv
Rows: 139 Columns: 13


,record_number,group_name,group_type,district,constituency,ward,zone,contact_person,venture_type,sector,source_id,year,cdf_category
0,1,Tubalange Women's Club,Women,Mumbwa,Mumbwa,Makebo,Kashinka,Tana Choongo,Village Banking,Finance,cdf_grants_mumbwa_2025,2025,grants
1,2,Tusole Women's Club,Women,Mumbwa,Mumbwa,Makebo,Mukanda,Eneless Sinkala,Village Banking,Finance,cdf_grants_mumbwa_2025,2025,grants
2,3,Minex Multi-Purpose Co-operative Society Limited,Community,Mumbwa,Mumbwa,Makebo,Muleke,Choolwe Hanungu,Keeping Chickens,Livestock,cdf_grants_mumbwa_2025,2025,grants
3,4,Mapesho Women Club,Women,Mumbwa,Mumbwa,Makebo,Muleke,Nyangulu Jessy,Gardening,Agriculture,cdf_grants_mumbwa_2025,2025,grants
4,5,Chabota Concession Women's Club,Women,Mumbwa,Mumbwa,Makebo,Big Concess,Ruth Choka,Poultry,Livestock,cdf_grants_mumbwa_2025,2025,grants



db-unza26-csc4792-mumbwa_town_council_cdf_not_approved_community_projects.csv
Rows: 135 Columns: 19


,record_number,project_name,project_description,sector,type_of_project,district,constituency,ward,zone,project_site_location,distance_km,year_funded,work_package,scope_of_works,status,comments,source_id,year,cdf_category
0,1,Completion of a Health post,Completion of a Health post at Kakumbi Village,Health,Completion,NaN,Mumbwa,NaN,NaN,Kakumbi Vilaage,NaN,NaN,NaN,NaN,Slab Level,Not in IDP,cdf_not_approved_projects_mumbwa_2025,2025,not_approved_community_projects
1,2,Contruction of 1*3 Classroom Block,Contruction of 1*3 Classroom Block at Mabele Primary School,Education,Construction,NaN,Mumbwa,NaN,NaN,Mabele Primary School,NaN,NaN,NaN,NaN,NaN,Not enough funds.,cdf_not_approved_projects_mumbwa_2025,2025,not_approved_community_projects
2,3,Water Reticulation System,Water Reticulation System at Nambala Rural Health Post,Water,Reticulation,NaN,Mumbwa,NaN,NaN,Nambala Rural Health Post,NaN,NaN,NaN,NaN,NaN,Not enough funds.,cdf_not_approved_projects_mumbwa_2025,2025,not_approved_community_projects
3,4,Construction of 1*3 Classroom Block,Construction of 1*3 Classroom Block at Kankunka,Education,Construction,NaN,Mumbwa,NaN,NaN,Kankunka,NaN,NaN,NaN,NaN,NaN,APPROVED,cdf_not_approved_projects_mumbwa_2025,2025,not_approved_community_projects
4,5,Construction of 1*3 Classroom Block,Construction of 1*3 Classroom Block at Mululi Primary School,Education,Construction,NaN,Mumbwa,NaN,NaN,Mululi Primary School,NaN,NaN,NaN,NaN,NaN,Not enough funds.,cdf_not_approved_projects_mumbwa_2025,2025,not_approved_community_projects



db-unza26-csc4792-mumbwa_town_council_cdf_secondary_bursaries.csv
Rows: 787 Columns: 18


,record_number,pupil_name,province,district,constituency,ward,zone,gender,date_of_birth,grade,grade_started_on_bursary,new_grade_2025,school_name,school_location,status,source_id,year,cdf_category
0,1.0,Phiri Gift James,NaN,Mumbwa,Mumbwa,Mupona,Welfare,M,27/10/2009,8,NaN,NaN,Mumbwa Secondary,Mumbwa,NaN,cdf_secondary_bursaries_mumbwa_2025,2025,secondary_bursaries
1,2.0,Miyanda Lweendo,NaN,Mumbwa,Mumbwa,Mupona,Makasa,M,06/10/2007,11,NaN,NaN,Mumbwa Secondary,Mumbwa,NaN,cdf_secondary_bursaries_mumbwa_2025,2025,secondary_bursaries
2,3.0,Miyanda Lushomo,NaN,Mumbwa,Mumbwa,Mupona,Makasa,M,27/01/2013,9,NaN,NaN,Mumbwa Secondary,Mumbwa,NaN,cdf_secondary_bursaries_mumbwa_2025,2025,secondary_bursaries
3,4.0,Mufungulwa Sharon,NaN,Mumbwa,Mumbwa,Mupona,Welfare,M,06/09/2012,10,NaN,NaN,Nambala Secondary,Mumbwa,NaN,cdf_secondary_bursaries_mumbwa_2025,2025,secondary_bursaries
4,5.0,Joel Jabulani Musindo,NaN,Mumbwa,Mumbwa,Mupona,Welfare,M,23/06/2009,10,NaN,NaN,Nambala Secondary,Mumbwa,NaN,cdf_secondary_bursaries_mumbwa_2025,2025,secondary_bursaries



db-unza26-csc4792-mumbwa_town_council_cdf_skills_bursaries.csv
Rows: 780 Columns: 16


,record_number,student_name,nrc_no,province,district,constituency,ward,zone,gender,course_or_skill,skill_level,programme_duration,institution,source_id,year,cdf_category
0,1.0,Suwilanji Phiri,NaN,NaN,Mumbwa,Mumbwa,Mupona,Council P:,F,Food and Nutrition,Diploma,36 Months,Natural Resources Development College (NRDC),cdf_skills_bursaries_mumbwa_2025,2025,skills_bursaries
1,2.0,Malambo Fenny Fatima,NaN,NaN,Mumbwa,Mumbwa,Mupona,Bulungu,F,Aeronautical Electronics Engineering,Diploma,36 Months,Zambia Air Services Training Institute,cdf_skills_bursaries_mumbwa_2025,2025,skills_bursaries
2,3.0,Abigail Mukobeko,NaN,NaN,Mumbwa,Mumbwa,Mupona,Bulungu,F,Computer Studies,Certificate,12 Months,Youth Resources Centre,cdf_skills_bursaries_mumbwa_2025,2025,skills_bursaries
3,4.0,Memory Mukandawire,NaN,NaN,Mumbwa,Mumbwa,Mupona,Bulungu,F,General Agriculture,Certificate,12 Months,Mumbwa Youth Resources Centre,cdf_skills_bursaries_mumbwa_2025,2025,skills_bursaries
4,5.0,Jimmy Kafwabwe,NaN,NaN,Mumbwa,Mumbwa,Mupona,Bulungu,M,General Agriculture,Certificate,12 Months,Hope College of Education,cdf_skills_bursaries_mumbwa_2025,2025,skills_bursaries


##  Basic Validation Checks

This final check confirms that final datasets exist, contain rows, and do not contain exact duplicate records.


In [18]:
validation_rows = []

for csv_file in final_files:
    df = pd.read_csv(csv_file, sep="|")
    validation_rows.append({
        "file": csv_file.name,
        "rows": len(df),
        "columns": len(df.columns),
        "duplicate_rows": int(df.duplicated().sum()),
        "empty_columns": int(df.isna().all().sum()),
    })

validation_df = pd.DataFrame(validation_rows)
validation_df


,file,rows,columns,duplicate_rows,empty_columns
0,db-unza26-csc4792-mumbwa_town_council_cdf_community_projects.csv,30,19,0,2
1,db-unza26-csc4792-mumbwa_town_council_cdf_grants.csv,139,13,0,0
2,db-unza26-csc4792-mumbwa_town_council_cdf_not_approved_community_projects.csv,135,19,0,4
3,db-unza26-csc4792-mumbwa_town_council_cdf_secondary_bursaries.csv,787,18,0,0
4,db-unza26-csc4792-mumbwa_town_council_cdf_skills_bursaries.csv,780,16,0,0
